# J4 — La mesure, et la révélation

**Séance 4.** Un modèle ne se juge pas sur un chiffre. Il se juge sur **une décision** : qui
alerte-t-on, et que coûte une erreur ? Aujourd’hui, on descend du score jusqu’au coût. On regarde
aussi ce que le classement devient quand on le mesure sur ce qui compte.

> **La règle :** une métrique n’est pas vraie ou fausse, elle est **adaptée ou non** à la décision
> qu’elle doit éclairer. Choisir la métrique, c’est déjà décider.

**Les chiffres du texte** sont ceux des cellules : si une cellule en donne un autre, c’est la
cellule qui a raison.

**Le fil du carnet.** Trois couches, dans cet ordre : la **perte**, ce que le modèle optimise
pendant l’apprentissage ; la **métrique**, ce que le rapport affiche ; le **coût**, ce que la
décision paie sur le terrain. On part de la métrique, on remonte à la perte, on finit par
le coût. Chaque section dit à quelle couche elle appartient.

**Ce que le devoir 2 vous demandera :** vos six métriques recalculées depuis **votre** matrice.
La courbe du coût et votre seuil. L’écart entre la métrique affichée et le coût réel. Et le
tableau « quand ne pas faire de ML ».

**La table de travail n’est pas reconstruite ici** : la mise en place est **extraite** de
`notebook_J1.ipynb` et de `notebook_J3.ipynb`, qui restent les sources uniques.


## Mise en place (fournie)

Ces cellules sont **extraites** des carnets précédents. Exécutez-les de haut en bas, sans rien y
modifier. Elles apportent les données, la table de travail, la référence, le découpage et le
modèle. Elles apportent aussi les baselines — dont la **barre de bruit** que la séance 3 a
posée.

Les `random_state=0` que vous verrez sont la **graine du tirage au sort** (expliquée en J1 §5).
Le même nombre redonne le même découpage, donc la même mesure.

**Deux mots avant de commencer**, parce que tout le carnet en dépend. Le **positif** est
**l’absence du patient** : c’est la classe rare. Et **alerter**, c’est **annoncer une absence**.

**Ce que chaque cas déclenche, concrètement :**

- une **alerte juste** (absence annoncée, patient bien absent) libère le créneau : il peut être
  réattribué à quelqu’un qui attend ;
- une **fausse alerte** (absence annoncée, patient venu) bloque le créneau pour rien et coûte
  **25 unités de coût (u.)** ;
- une **absence manquée** (patient absent, rien d’annoncé) laisse le créneau vide : **100 u.**

Ces deux montants sont **fixés par l’énoncé**. Dans un vrai projet, la direction
de la clinique les donnerait — et ce carnet ne les discute pas, il les utilise.


In [ ]:
# le fil du carnet, en une image — trois couches, du réglage à la facture
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9.6, 2.0))
ax.axis('off')
niveaux = [('1 · la perte', 'ce que le modèle optimise\npendant l’apprentissage', '#2b6cb0'),
           ('2 · la métrique', 'ce que le rapport\naffiche', '#4a5568'),
           ('3 · le coût', 'ce que la décision\npaie sur le terrain', '#b5651d')]
for i, (titre, texte, couleur) in enumerate(niveaux):
    x = 0.04 + i * 0.34
    ax.text(x, 0.50, titre + '\n' + texte, ha='left', va='center', fontsize=11,
            color='white', bbox=dict(boxstyle='round,pad=0.6', facecolor=couleur, edgecolor='none'))
    if i < 2:
        ax.annotate('', xy=(x + 0.315, 0.50), xytext=(x + 0.245, 0.50),
                    arrowprops=dict(arrowstyle='->', color='#2d3748', lw=1.5))
plt.tight_layout()
plt.show()


### Les outils


In [ ]:
# les outils
# Aucun import en bloc au début : chaque module entre dans le carnet au moment de son premier
# usage. On voit ainsi à quoi il sert, au lieu de le deviner dans une liste.

# `display` : un tableau s'affiche proprement dans un carnet, et reste lisible ailleurs.
# C'est par `display` qu'on montre TOUTES les tables de ce carnet — jamais un print de texte.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


### Les données


In [ ]:
import os
import pandas as pd
# où sont les fichiers — on CHERCHE `train.csv`, on ne suppose pas un chemin
# (Kaggle a déjà déplacé ses points de montage : un chemin recopié se périme, une recherche non)
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie"])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"

print("le dossier des données trouvé :", DOSSIER)


In [ ]:
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"

print("les deux tableaux \n")
print("train :", train.shape, "(lignes, colonnes)")
print("test  :", test.shape, "(lignes, colonnes)")

print("les cinq premières lignes \n")
display(train.head())


### La table de travail


In [ ]:
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La table de travail : colonnes écartées retirées, catégorielles déclarées."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)

print("la table de travail \n")
print("colonnes conservées :", X.shape[1])
print("colonnes écartées (cardinalité trop forte) :", TROP_FIN)
print("colonnes catégorielles :", CAT)


### La cible, et la référence


In [ ]:
print("la distribution de la cible \n")
repartition = pd.DataFrame({"lignes": [len(train), int(train[CIBLE].sum())],
                            "part (%)": [100.0, round(100 * train[CIBLE].mean(), 2)]},
                           index=["toutes", "absents"])
display(repartition)

reference = 100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean())
print(f"la référence (répondre toujours la classe majoritaire) : {reference:.2f} %")


In [ ]:
# `y` entre ici ; la référence a déjà été calculée avec la distribution ci-dessus —
y = train[CIBLE]


### Le découpage, le modèle, et les baselines


In [ ]:
# les baselines, sur le même jeu de validation — et la barre qui décide
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

# 1) la réponse constante : le concurrent le plus bête, et le plus dur à battre
constante = 100 * max(yval.mean(), 1 - yval.mean())

# 2) la règle du délai : « j'annonce une absence si le délai dépasse N jours ».
#    N se choisit sur les lignes d'apprentissage, pas sur celles de contrôle.
delai_max = int(Xtr['delai_jours'].max())
taux_par_seuil = {s: float(((Xtr['delai_jours'] > s).astype(int) == ytr).mean())
                  for s in (0, 1, 3, 7, 14, 30) if s < delai_max}
seuil_appris = max(taux_par_seuil, key=taux_par_seuil.get)
regle = (Xval['delai_jours'] > seuil_appris).astype(int)

# et la règle qui ne dirait jamais rien ? c'est la réponse constante, au point près
silence = (Xval['delai_jours'] > delai_max).astype(int)

# 3) la famille linéaire : elle a besoin de nombres, on lui en prépare — la recette des
#    séances précédentes (newton-cholesky : le solveur qui converge sur cette table)
X_n = pd.get_dummies(X, columns=CAT).fillna(0)   # 0 = trou comblé ; le linéaire ne lit pas les NaN
Xtr_n, Xval_n = X_n.loc[Xtr.index], X_n.loc[Xval.index]
lineaire = LogisticRegression(solver='newton-cholesky').fit(Xtr_n, ytr)

# 4) le modèle du cours, celui des séances précédentes
modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)

# la barre : l'erreur-type de l'exactitude sur CE jeu de validation
barre = 100 * ((constante / 100) * (1 - constante / 100) / len(yval)) ** 0.5

baselines = pd.DataFrame([
    {'baseline': 'toujours présent',
     'exactitude (%)': round(constante, 2)},
    {'baseline': f'règle du délai — absent si le délai dépasse {seuil_appris} j',
     'exactitude (%)': round(100 * (regle == yval).mean(), 2)},
    {'baseline': 'absent si ce patient a déjà manqué un rendez-vous',
     'exactitude (%)': round(100 * ((Xval['taux_absence_passe'] > 0).astype(int) == yval).mean(), 2)},
    {'baseline': 'la famille linéaire',
     'exactitude (%)': round(100 * lineaire.score(Xval_n, yval), 2)},
    {'baseline': 'le modèle du cours',
     'exactitude (%)': round(100 * modele.score(Xval, yval), 2)},
])
display(baselines)

ecart = 100 * modele.score(Xval, yval) - constante
print(f'→ le modèle dépasse la réponse constante de {ecart:+.2f} point,')
print(f'  soit {ecart / barre:.1f} erreurs-types de ce jeu de validation ({barre:.2f} point).')
print(f'  la règle du délai, elle, fait {100 * (regle == yval).mean():.2f} % : elle alerte sur')
print(f'  {int(regle.sum())} lignes et perd contre la réponse constante.')
print(f'  et la règle qui ne dirait jamais rien ferait {100 * (silence == yval).mean():.2f} % —')
print('  exactement la réponse constante. Une règle en place se mesure avant d’être opposée')
print('  à un modèle ; ce que cela dit d’une exactitude reste ouvert.')


### Le rituel : annoncer avant de mesurer


In [ ]:
# l'horloge entre ici : c'est elle qui rend une prédiction vérifiable
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté.
       `noter(...)` refuse ensuite tout essai sans prédiction : l'ordre n'est pas demandé, il est
       appliqué. Ce que cette fonction garantit, exactement : une prédiction EXISTAIT avant que la
       ligne du tableau soit écrite. Elle ne date pas la mesure — le score lui arrive déjà calculé.
       Pour la garantie complète, mesurez dans `essai(...)`, qui date la mesure lui-même."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p["essai"] != essai]
    PREDICTIONS.append({"essai": essai, "attendu": attendu, "raison": raison,
                        "écrit à": datetime.now().strftime("%H:%M:%S")})
    display(pd.DataFrame(PREDICTIONS))

def essai(nom, mesure, changement="", attendu=""):
    """Le chemin STRICT : la mesure est prise ICI, donc l'ordre (prédiction puis mesure) est prouvé.
       `mesure` est une fonction sans argument qui rend le score, par exemple
       `lambda: modele.score(Xval, yval)`. Refuse de mesurer sans prédiction."""
    prevu = [p for p in PREDICTIONS if p["essai"] == nom]
    if not prevu:
        print(f"refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d'abord.")
        return None
    score = 100 * mesure()
    noter(nom, changement or "voir la question", attendu or prevu[0]["attendu"], score)
    return score

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. `attendu` : un signe et une raison, jamais une valeur.
       Ré-exécuter une cellule REMPLACE la ligne de même `hypothèse` : elle ne la double pas.

       REFUSE d'enregistrer un essai non annoncé — c'est la règle du cours, appliquée.
       Ce qu'elle garantit, et rien de plus : une prédiction existait AVANT que cette ligne soit
       écrite. Elle ne date pas la mesure — d'où `essai(...)` pour la garantie complète."""
    prevu = [p for p in PREDICTIONS if p["essai"] == hypothese]
    if not prevu:
        print(f"  refusé : aucune prédiction pour « {hypothese} ».")
        print("  Écrivez prevoir(...) d'abord — l'ordre est vérifié, pas seulement demandé.")
        return
    ligne = {"hypothèse": hypothese, "changement": changement,
             "attendu": attendu, "mesuré": round(score, 2),
             "prédiction écrite avant ?": "oui, à " + prevu[0]["écrit à"]}
    ESSAIS[:] = [e for e in ESSAIS if e["hypothèse"] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))


In [ ]:
# les avertissements des bibliothèques sont coupés : ils ne sont pas la leçon
import warnings; warnings.filterwarnings('ignore')


### La matrice de confusion

**Couche — la métrique.** La matrice est le socle de toutes les mesures qui suivent.

Une exactitude ne dit pas **où** le modèle se trompe. Deux modèles qui se trompent à l’opposé
affichent le même chiffre. La matrice de confusion garde cette information : deux réponses
possibles, deux erreurs possibles, quatre cases.

**Les quatre noms, en clair.**

- **Vrai positif (VP)** : le patient ne venait pas, et le modèle l’a annoncé. Juste et utile.
- **Faux positif (FP)** : le patient venait, et le modèle a annoncé son absence. C’est la **fausse alerte**.
- **Faux négatif (FN)** : le patient ne venait pas, et le modèle a annoncé sa présence. C’est l’**absence manquée**.
- **Vrai négatif (VN)** : le patient venait, et le modèle l’a annoncé présent. Juste et sans effet.

| Ce qui s’est passé | Le modèle annonce une absence (positif) | Le modèle annonce une présence (négatif) |
|---|---|---|
| **Le patient ne vient pas** | **Vrai positif (VP)** — le créneau est libéré | **Faux négatif (FN)** — le créneau reste vide |
| **Le patient vient** | **Faux positif (FP)** — le créneau est bloqué pour rien | **Vrai négatif (VN)** — rien ne change |

**Lignes = ce qui s’est passé. Colonnes = ce qui a été annoncé.** On compte les quatre cases à
la main. Puis on les vérifie contre la bibliothèque. Enfin on les colorie : verts les deux
justes, rouges les deux erreurs.


In [ ]:
# les quatre cases, comptées à la main sur le jeu de contrôle
predit = modele.predict(Xval)

VP = int(((predit == 1) & (yval == 1)).sum())   # annoncé absent, et absent
FP = int(((predit == 1) & (yval == 0)).sum())   # annoncé absent, et venu : la fausse alerte
FN = int(((predit == 0) & (yval == 1)).sum())   # annoncé présent, et absent : l’absence manquée
VN = int(((predit == 0) & (yval == 0)).sum())   # annoncé présent, et venu

assert VP + FP + FN + VN == len(yval)

# et la probabilité, qui servira à déplacer le seuil : c’est elle qui décide, pas le 0/1
probabilites = modele.predict_proba(Xval)[:, 1]

# les quatre mêmes cases, retrouvées par la bibliothèque — le contrôle, pas la confiance
from sklearn.metrics import confusion_matrix
confirme = confusion_matrix(yval, predit).ravel()          # ordre : VN, FP, FN, VP
assert (VN, FP, FN, VP) == tuple(int(v) for v in confirme), 'les quatre cases ne concordent pas'
print(f'les quatre cases, vérifiées : VP {VP} · FP {FP} · FN {FN} · VN {VN}')
print(f'→ {FP + FN} erreurs en tout : {FP} fausses alertes, {FN} absences manquées.')


In [ ]:
# la même matrice, coloriée — c’est la forme qu’on lit d’un coup d’œil
import numpy as np
import matplotlib.pyplot as plt

valeurs = np.array([[VN, FP], [FN, VP]])
noms = np.array([['Vrai négatif (VN)', 'Faux positif (FP)'],
                 ['Faux négatif (FN)', 'Vrai positif (VP)']])
justes = np.array([[1, 0], [0, 1]])          # 1 = juste, 0 = erreur

fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.imshow(justes, cmap='RdYlGn', vmin=0, vmax=1)
for i in range(2):
    for j in range(2):
        ax.text(j, i, noms[i, j] + '\n' + f'{valeurs[i, j]:,}'.replace(',', ' '),
                ha='center', va='center', fontsize=10)
ax.set_xticks([0, 1], ['annoncé : il vient', 'annoncé : il ne vient pas'])
ax.set_yticks([0, 1], ['en réalité : il vient', 'en réalité : il ne vient pas'])
ax.set_title('la matrice coloriée : vert = juste, rouge = erreur')
plt.tight_layout()
plt.show()


### Les six métriques, depuis la même matrice

**Couche — la métrique.** Six questions posées au même tableau.

Le même tableau donne six chiffres. Chacun répond à une question différente, et chacun a un
type de tâche où il est le bon outil.

**Le schéma d’abord** : pour chaque métrique, les cases qui comptent. En bleu foncé, les cases
du numérateur ; en bleu clair, celles du dénominateur.


In [ ]:
# une planche de six panneaux : quelles cases compte chaque métrique
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

noms_cases = {(0, 0): 'VN', (0, 1): 'FP', (1, 0): 'FN', (1, 1): 'VP'}
panneaux = [
    ('exactitude', '(VP + VN) / tout', {(0, 0), (0, 1), (1, 0), (1, 1)}, {(0, 0), (0, 1), (1, 0), (1, 1)}),
    ('exactitude équilibrée', '(rappel + spécificité) / 2', {(0, 0), (1, 1)}, {(0, 0), (0, 1), (1, 0), (1, 1)}),
    ('précision', 'VP / (VP + FP)', {(1, 1)}, {(1, 1), (0, 1)}),
    ('rappel', 'VP / (VP + FN)', {(1, 1)}, {(1, 1), (1, 0)}),
    ('spécificité', 'VN / (VN + FP)', {(0, 0)}, {(0, 0), (0, 1)}),
    ('F1', '2·VP / (2·VP + FP + FN)', {(1, 1)}, {(1, 1), (0, 1), (1, 0)}),
]
figure, axes = plt.subplots(2, 3, figsize=(10.6, 5.2))
for axe, (titre, formule, numerateur, denominateur) in zip(axes.ravel(), panneaux):
    axe.set_xlim(0, 2); axe.set_ylim(0, 2); axe.axis('off')
    for (i, j) in [(0, 0), (0, 1), (1, 0), (1, 1)]:
        x, ligne_y = j, 1 - i
        couleur = '#1f4e79' if (i, j) in numerateur else ('#a8c6e0' if (i, j) in denominateur else '#f0f0f0')
        axe.add_patch(Rectangle((x, ligne_y), 1, 1, facecolor=couleur, edgecolor='white', linewidth=2))
        axe.text(x + 0.5, ligne_y + 0.5, noms_cases[(i, j)], ha='center', va='center', fontsize=11)
    axe.set_title(titre + '\n' + formule, fontsize=9)
figure.suptitle('les cases du numérateur (foncé) et du dénominateur (clair)', fontsize=10)
plt.tight_layout()
plt.show()


| Métrique | Formule | En mots | La bonne métrique quand… | Exemple évident |
|---|---|---|---|---|
| **exactitude** | (VP + VN) / tout | la part de bonnes réponses | les deux classes sont équilibrées | compter les pièces bonnes et mauvaises sur une chaîne |
| **exactitude équilibrée** | (rappel + spécificité) / 2 | la moyenne des deux taux, un par classe | les classes sont très déséquilibrées | une maladie rare, où l’exactitude brute est trompeuse |
| **précision** | VP / (VP + FP) | sur les alertes lancées, la part qui étaient justes | une alerte coûte cher | un contrôle douanier, où chaque fouille coûte du temps |
| **rappel** | VP / (VP + FN) | sur les absences réelles, la part trouvée | rater coûte plus cher qu’alerter | un dépistage médical : deux examens inutiles valent mieux qu’un cas raté |
| **spécificité** | VN / (VN + FP) | sur les patients venus, la part que l’on n’a pas alertée à tort | les présences sont la majorité | un filtre anti-spam qui ne doit jamais bloquer un vrai courriel |
| **F1** | 2 × précision × rappel / (précision + rappel) | la moyenne harmonique de la précision et du rappel | on ne sait pas laquelle des deux erreurs payer | comparer deux modèles sur un problème nouveau |


### La démonstration, métrique par métrique

Nos quatre cases : **VP = 580**, **FP = 353**, **FN = 4 891**, **VN = 24 155**, sur 29 979 lignes de
contrôle. Chaque métrique est recalculée à la main sur ces quatre nombres.

**Exactitude.**

$$\text{exactitude} = \frac{\text{bonnes réponses}}{\text{toutes les lignes}}$$

Ce qu'elle calcule : la part de lignes où le modèle a vu juste, toutes classes confondues.
Quand c'est la bonne métrique : les deux classes sont équilibrées et une erreur en vaut une autre.
Sur notre matrice : (580 + 24 155) / 29 979 = **82,51 %**. Ne rien faire vaut 81,75 % : l'écart
tient en 0,76 point.

**Exactitude équilibrée.**

$$\text{équilibrée} = \frac{1}{2}\left(\frac{\text{absences trouvées}}{\text{absences réelles}} + \frac{\text{présences non alertées}}{\text{présences réelles}}\right)$$

Ce qu'elle calcule : la moyenne des deux taux de réussite, un par classe — chaque classe pèse
autant, quelle que soit sa fréquence.
Quand c'est la bonne métrique : les classes sont très déséquilibrées et on ne veut pas que la
classe majoritaire décide seule.
Sur notre matrice : (10,60 % + 98,56 %) / 2 = **54,58 %**. L'exactitude affichait 82,51 % :
l'écart entre les deux est la mesure du déséquilibre.

**Précision.**

$$\text{précision} = \frac{\text{alertes justes}}{\text{alertes lancées}}$$

Ce qu'elle calcule : sur les alertes lancées, la part qui étaient justes.
Quand c'est la bonne métrique : une alerte coûte cher — on ne dérange pas pour rien.
Sur notre matrice : 580 / (580 + 353) = 580 / 933 = **62,17 %**. Sur cent alertes, soixante-deux
sont justes.

**Rappel.**

$$\text{rappel} = \frac{\text{absences trouvées}}{\text{absences réelles}}$$

Ce qu'elle calcule : sur les absences réelles, la part que le modèle a trouvées.
Quand c'est la bonne métrique : rater coûte plus cher qu'alerter.
Sur notre matrice : 580 / (580 + 4 891) = 580 / 5 471 = **10,60 %**. Sur dix absences, une est
trouvée : c'est le chiffre le plus dur de la séance.

**Spécificité.**

$$\text{spécificité} = \frac{\text{présences non alertées}}{\text{présences réelles}}$$

Ce qu'elle calcule : sur les patients venus, la part que l'on n'a pas alertée à tort.
Quand c'est la bonne métrique : les présences sont la majorité et on veut les laisser tranquilles.
Sur notre matrice : 24 155 / (24 155 + 353) = 24 155 / 24 508 = **98,56 %**. Sur cent patients
venus, un est alerté à tort.

**F1.**

$$F_1 = \frac{2 \cdot \text{précision} \cdot \text{rappel}}{\text{précision} + \text{rappel}}$$

où P est la précision et R le rappel. Ce qu'il calcule : la moyenne **harmonique** des deux.
Quand c'est la bonne métrique : on ne sait pas laquelle des deux erreurs payer. On veut un seul
chiffre, qui ne laisse pas un des deux s'effondrer.
Sur notre matrice : 2 × 0,6217 × 0,1060 / (0,6217 + 0,1060) = **18,11 %**. La moyenne ordinaire
donnerait 36 % : l'écart est la punition du déséquilibre entre précision et rappel.


### Trois situations où la métrique change tout

La même matrice, trois mondes. Ce qui décide de la métrique à suivre, ce n'est pas le modèle :
c'est le prix des deux erreurs.

**Une carte bancaire bloquée à tort.** Une banque surveille les paiements pour détecter la
fraude. Chaque alerte bloque la carte du client : il appelle, attend, parfois change de banque.
Ici, une **fausse alerte coûte cher**. On suit donc la **précision** : sur cent alertes, combien
sont de vraies fraudes. Le rappel peut rester bas — rater une fraude coûte, mais bloquer des
centaines de clients honnêtes coûte plus encore. Le seuil se règle **haut**, et l'équipe peut
traiter à la main le peu d'alertes qui restent.

**Un dépistage qui rate un cas.** Un test dépiste une maladie grave. Chaque cas manqué peut
coûter la vie ; une alerte à tort déclenche un examen supplémentaire, désagréable mais sans
danger. Ici, une **absence manquée coûte cher**. On suit donc le **rappel** : sur cent malades,
combien sont détectés. Les fausses alertes sont acceptées, et le seuil se règle **bas**.

**Un dossier de bourse vérifié à la main.** Une fondation à Cotonou reçoit 1 000 demandes.
Un modèle repère les dossiers à vérifier, puis un agent tranche. Vérifier un dossier coûte 30 u. ;
une bourse versée à tort en coûte 100 u. Sur ces demandes, 120 sont inéligibles. Au seuil actuel,
le modèle envoie 200 dossiers. Cent sont inéligibles : la précision vaut 50 %. Vingt passent
inaperçus : le rappel vaut 83 %. Baisser le seuil enverrait 350 dossiers. Le rappel monterait à
98 %, mais 232 dossiers éligibles seraient vérifiés pour rien. Le coût total passerait de
8 000 u. à 10 700 u. Ici, on surveille la **précision** : chaque alerte coûte une vérification.
Le seuil actuel reste le bon.

**Et notre clinique ?** Elle est entre les deux : une absence manquée coûte 100 u., une fausse
alerte 25 u. Le coût est asymétrique, mais pas au point d'un dépistage. La décision se prend donc
sur le **coût**. La précision et le rappel, eux, se lisent **ensemble** : la suite de ce carnet
chiffre ce compromis, seuil par seuil.

**La règle à retenir.** Avant de choisir une métrique, posez une seule question : laquelle des
deux erreurs, et à quel prix ? La réponse donne la métrique, puis le seuil.


In [ ]:
# les six métriques, recalculées depuis les quatre cases — jamais depuis une autre source
exactitude = (VP + VN) / (VP + FP + FN + VN)
rappel = VP / (VP + FN)               # la part des patients absents qu’on a trouvés
specificite = VN / (VN + FP)          # la part des patients venus qu’on n’a pas alertés à tort
# la part d’alertes justes — et elle n’existe pas si le modèle n’alerte jamais : 0/0 n’est pas 0
precision = VP / (VP + FP) if (VP + FP) else float('nan')
f1 = 2 * precision * rappel / (precision + rappel)   # la moyenne harmonique
equilibree = (rappel + specificite) / 2

lecture = pd.DataFrame([
    {'métrique': 'exactitude', 'formule': '(VP + VN) / tout', 'valeur (%)': round(100 * exactitude, 2),
     'lecture': f'{100 * exactitude:.0f} bonnes réponses sur 100 lignes'},
    {'métrique': 'exactitude équilibrée', 'formule': '(rappel + spécificité) / 2', 'valeur (%)': round(100 * equilibree, 2),
     'lecture': f'{100 * equilibree:.0f} : la moyenne des deux taux, un par classe'},
    {'métrique': 'précision', 'formule': 'VP / (VP + FP)', 'valeur (%)': round(100 * precision, 2),
     'lecture': f'sur 100 alertes, {100 * precision:.0f} justes'},
    {'métrique': 'rappel', 'formule': 'VP / (VP + FN)', 'valeur (%)': round(100 * rappel, 2),
     'lecture': f'sur 100 absences réelles, {100 * rappel:.0f} trouvées'},
    {'métrique': 'spécificité', 'formule': 'VN / (VN + FP)', 'valeur (%)': round(100 * specificite, 2),
     'lecture': f'sur 100 patients venus, {100 - 100 * specificite:.0f} alerté à tort'},
    {'métrique': 'F1', 'formule': '2·VP / (2·VP + FP + FN)', 'valeur (%)': round(100 * f1, 2),
     'lecture': f'{100 * f1:.0f} : le compromis entre précision et rappel'},
])
display(lecture)

# et les six chiffres côte à côte : c’est la figure qui montre qu’ils ne mesurent pas la même chose
figure, axe = plt.subplots(figsize=(8, 3.2))
couleurs = ['#c0392b' if m == 'exactitude' else '#8a9ba8' for m in lecture['métrique']]
axe.barh(lecture['métrique'], lecture['valeur (%)'], color=couleurs)
axe.set_xlim(0, 100)
axe.set_xlabel('valeur (%)')
axe.set_title('six chiffres, six questions — depuis la même matrice')
for position, v in enumerate(lecture['valeur (%)']):
    axe.text(v + 1.5, position, f'{v:.1f} %', va='center', fontsize=9)
plt.tight_layout()
plt.show()
print('→ l’exactitude compte les bons pronostics ; le rappel, les absences trouvées ;')
print('  la précision, la part d’alertes justes. Choisir l’une, c’est choisir ce qu’on veut.')


### Quand l’exactitude répond à une autre question

**Couche — la métrique.** Le piège de celle qui ne dit rien de la décision.

La cible est rare : un patient sur cinq est absent. Répondre « il vient » à tout le monde donne
donc un score flatteur, sans lire une seule colonne. Trois stratégies, trois colonnes : le même
tableau montre ce que chaque chiffre récompense.


In [ ]:
# trois stratégies, trois métriques — ce que chacune récompense
def trois(decision):
    justes = int(((decision == 1) & (yval == 1)).sum())
    faux = int(((decision == 1) & (yval == 0)).sum())
    manques = int(((decision == 0) & (yval == 1)).sum())
    vrais = int(((decision == 0) & (yval == 0)).sum())
    return {'exactitude (%)': round(100 * (justes + vrais) / len(yval), 2),
            'précision (%)': round(100 * justes / (justes + faux), 2) if justes + faux else float('nan'),
            'rappel (%)': round(100 * justes / (justes + manques), 2)}

tableau = pd.DataFrame([
    {'stratégie': 'n’alerte jamais', **trois(np.zeros(len(yval), dtype=int))},
    {'stratégie': 'alerte toujours', **trois(np.ones(len(yval), dtype=int))},
    {'stratégie': 'le modèle', **trois(predit)},
])
display(tableau)

jamais = tableau.loc[0]; toujours = tableau.loc[1]; modele_ligne = tableau.loc[2]
ex_jamais, rap_toujours, prec_toujours = jamais['exactitude (%)'], toujours['rappel (%)'], toujours['précision (%)']
ex_modele, rap_modele = modele_ligne['exactitude (%)'], modele_ligne['rappel (%)']
print(f'→ ne rien faire : {ex_jamais:.2f} % d’exactitude et 0 absence trouvée.')
print(f'  alerter toujours : {rap_toujours:.0f} % de rappel, mais {prec_toujours:.2f} % de précision')
print('  — une alerte sur cinq est juste.')
print(f'  le modèle : {ex_modele:.2f} % d’exactitude, {rap_modele:.2f} % de rappel.')
print('  Un chiffre qui récompense le modèle qui ne fait rien n’est pas une mesure : c’est un piège.')


### Ce que le rang mesure, et ce que la perte mesure

**Couches — la métrique et la perte.** Deux façons de juger, et ce qu’elles ignorent.

**Le prérequis**, parce que tout le reste en dépend : le modèle ne rend pas un oui ou un non,
il rend une **probabilité**. Un **seuil** transforme cette probabilité en décision. Les métriques
de rang jugent la probabilité sans choisir de seuil ; les métriques de perte jugent la
probabilité elle-même.

**La courbe ROC** croise le taux de vrais positifs (le rappel) et le taux de faux positifs
`FP / (FP + VN)`. Sa diagonale est le hasard. **L’aire sous la ROC (ROC-AUC)** se lit ainsi : la
chance qu’un patient absent tiré au hasard soit mieux classé qu’un patient venu tiré au hasard.
0,5 = le hasard, 1 = parfait.

**La courbe précision/rappel (PR)** croise la précision et le rappel. Son plancher n’est pas une
diagonale : c’est le **taux de base**, la part d’absents. Une PR-AUC se lit donc **en multiple
de son plancher** : la cellule imprime ce rapport, et c’est lui qui parle.

**Pourquoi la ROC flatte une cible rare** : son taux de faux positifs divise les fausses alertes
par le nombre de patients venus, qui se compte en dizaines de milliers. La précision, elle, les
divise par le nombre d’alertes, qui varie fortement avec le seuil. Le même nombre d’erreurs
moins dans la ROC, et la table qui suit le chiffre.


In [ ]:
# les deux courbes, côte à côte, et les seuils marqués dessus
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

faux_positifs, vrais_positifs, _ = roc_curve(yval, probabilites)
precisions, rappels, _ = precision_recall_curve(yval, probabilites)
figure, (gauche, droite) = plt.subplots(1, 2, figsize=(10.6, 3.9))

gauche.plot(faux_positifs, vrais_positifs, color='#2b6cb0', linewidth=2)
gauche.plot([0, 1], [0, 1], color='#a0aec0', linestyle='--', linewidth=1)
gauche.set_xlabel('taux de faux positifs'); gauche.set_ylabel('taux de vrais positifs (rappel)')
gauche.set_title(f'ROC — aire {roc_auc_score(yval, probabilites):.3f}')

droite.plot(rappels, precisions, color='#b5651d', linewidth=2)
droite.axhline(yval.mean(), color='#a0aec0', linestyle='--', linewidth=1)
droite.set_xlabel('rappel'); droite.set_ylabel('précision')
droite.set_title(f'PR — aire {average_precision_score(yval, probabilites):.3f} (plancher {yval.mean():.3f})')

for seuil in (0.20, 0.30, 0.50):
    alerte = probabilites > seuil
    justes = int((alerte & (yval == 1)).sum()); faux = int((alerte & (yval == 0)).sum())
    x_roc, y_roc = faux / (yval == 0).sum(), justes / (yval == 1).sum()
    x_pr, y_pr = justes / (yval == 1).sum(), justes / max(justes + faux, 1)
    gauche.plot(x_roc, y_roc, 'o', color='#2d3748')
    gauche.annotate(f'{seuil:.2f}', (x_roc, y_roc), textcoords='offset points', xytext=(5, 5), fontsize=8)
    droite.plot(x_pr, y_pr, 'o', color='#2d3748')
    droite.annotate(f'{seuil:.2f}', (x_pr, y_pr), textcoords='offset points', xytext=(5, 5), fontsize=8)
plt.tight_layout()
plt.show()
aire_pr = average_precision_score(yval, probabilites)
print(f'→ la PR-AUC vaut {aire_pr / yval.mean():.1f} fois le taux de base ({yval.mean():.3f}) :')
print('  c’est ce rapport qui parle, pas l’aire seule.')


In [ ]:
# le même nombre d’erreurs, vu par les deux métriques — c’est là que la ROC flatte
lignes = []
for seuil in np.round(np.arange(0.10, 0.71, 0.10), 2):
    alerte = (probabilites > seuil)
    justes = int((alerte & (yval == 1)).sum()); faux = int((alerte & (yval == 0)).sum())
    lignes.append({'seuil': seuil, 'alertes': int(alerte.sum()),
                   'précision (%)': round(100 * justes / max(justes + faux, 1), 1),
                   'taux de faux positifs (%)': round(100 * faux / (yval == 0).sum(), 2)})
display(pd.DataFrame(lignes))
venus = int((yval == 0).sum())
print(f'→ le taux de faux positifs divise par {venus:,} patients venus ;')
a_min, a_max = int(lignes[-1]['alertes']), int(lignes[0]['alertes'])
print(f'  la précision divise par les alertes — de {a_min} à {a_max} selon le seuil. Même erreur, poids différent.')

# et le plafond de la séance 3, relu en PR-AUC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
print('le même plafond, relu en PR-AUC')
for nom, famille, table in (
        ('arbres (boosting)', HistGradientBoostingClassifier(random_state=0), Xtr),
        ('un arbre unique', DecisionTreeClassifier(random_state=0, max_depth=8), Xtr_n),
        ('modèle linéaire', LogisticRegression(solver='newton-cholesky'), Xtr_n)):
    ajuste = famille.fit(table, ytr)
    cible = Xval if table is Xtr else Xval_n
    print(f'  {nom:<18} PR-AUC {average_precision_score(yval, ajuste.predict_proba(cible)[:, 1]):.4f}')
print('→ trois familles quasi égales en exactitude ne se valent pas en PR-AUC :')
print('  la métrique choisie décide du classement.')


### La perte : ce que le modèle optimise

**Couche — la perte.** Le chiffre que l’entraînement voit, et que le rapport ne montre jamais.

**Le principe, vu en séance 3.** Un modèle ajuste ses paramètres pour diminuer un nombre, la
**perte**. Elle n’apparaît jamais dans un rapport : elle sert pendant l’apprentissage. La séance 3
l’a montrée en train de descendre, à côté des scores.

**La log-loss, et ce qu’elle punit.** Elle vaut `−log(p)` pour la vraie classe. Elle punit
très fort une annonce confiante et fausse : pour un patient absent, annoncer 0,01 coûte 4,6.
Elle ne coûte presque rien à une annonce juste et confiante : annoncer 0,99, pour ce même
patient absent, coûte 0,01.

**Le linéaire descend la perte par itérations.** La régression logistique calcule la pente de la
log-loss sur tout l’échantillon, et descend d’un cran à chaque pas. Plus de pas, une perte plus
basse, jusqu’à un plateau.

**L’arbre, lui, ne descend rien.** Il n’a pas d’itérations : il découpe la table pour rendre
chaque feuille plus pure, et ce qu’il réduit est l’**impureté** (Gini ou entropie). Le réentraîner
sur les mêmes données redonne le même arbre.

**Le renforcement réunit les deux** : chaque arbre ajouté vise les erreurs que les précédents
laissent. Il fait donc baisser la même log-loss, arbre par arbre.

| Perte | Pour quoi | Ce qu’elle punit | Semble s’effondrer quand… |
|---|---|---|---|
| log-loss | une classe | une annonce confiante et fausse | les probabilités sont justes |
| erreur au carré | un nombre | un écart grand à la vérité | la prévision colle au réel |
| impureté (Gini) | construire un arbre | un morceau mélangé | les feuilles sont pures |


In [ ]:
# le linéaire descend sa perte par pas ; l’arbre, lui, n’a pas de pas
from sklearn.metrics import log_loss
from sklearn.linear_model import LogisticRegression
for pas in (1, 2, 3, 5, 8):
    m = LogisticRegression(solver='newton-cholesky', max_iter=pas).fit(Xtr_n, ytr)
    print(f'  pas {pas} · log-loss validation {log_loss(yval, m.predict_proba(Xval_n)[:, 1]):.4f}')
print('→ la perte du linéaire baisse avec les pas, puis se stabilise.')
print('  Un arbre unique s’entraîne en une fois : pas d’itérations à suivre.')


### La démonstration, perte par perte

**Log-loss (entropie croisée binaire).**

$$\text{log-loss} = -\frac{1}{N}\sum_{i=1}^{N}\left[y_i \log p_i + (1-y_i)\log(1-p_i)\right]$$
$$= -\frac{1}{N}\sum_{i=1}^{N}\log\left(\text{probabilité annoncée pour la vraie classe}\right)$$

où $y_i$ est la vraie classe et $p_i$ la probabilité annoncée pour la classe 1.
Ce qu'elle calcule : la qualité des **probabilités**, pas seulement la décision finale.
Quand : toute classification qui annonce une probabilité — c'est la perte des modèles du cours.
Sur notre matrice : **0,4139** pour le modèle, **0,4752** pour la probabilité constante.
Sur quatre patients (1, 1, 0, 0), avec les probabilités 0,90 · 0,05 · 0,80 · 0,30 : **1,2668**.
Ce chiffre est calculé à la main, puis vérifié contre la bibliothèque.

**Erreur au carré (régression).**

$$\text{erreur} = (\text{valeur vraie} - \text{valeur prévue})^2$$

Ce qu'elle calcule : l'écart réel entre la valeur prévue et la valeur vraie.
Quand : la cible est un **nombre**, pas une classe.
Sur notre cible numérique : prévoir 35 quand la vérité est 37 coûte 4 ; prévoir 30 coûte 49.
L'erreur au carré fait payer cher l'écart grand.

**Impureté (Gini), pour l'arbre.**

$$\text{Gini} = 1 - \sum_{k}\left(\text{part de la classe } k\right)^2$$

Ce qu'elle calcule : le mélange d'une feuille — 0 si la feuille est pure.
Quand : **construire** un arbre, pas juger un modèle.
Sur une feuille à 30 % d'absents et 70 % de présents : 1 − (0,3² + 0,7²) = **0,42** ; une feuille
pure vaut 0.

**À retenir.** On **entraîne** avec la perte — dérivable, sensible à la confiance. On **évalue**
avec la métrique — lisible. Une exactitude qui monte alors que la log-loss de validation remonte
est le signe d'un modèle trop confiant : le sur-apprentissage, vu sur deux chiffres.


In [ ]:
# la forme de la log-loss : ce qu’elle inflige à une annonce confiante
import numpy as np
import matplotlib.pyplot as plt

p = np.linspace(0.01, 0.99, 200)
figure, axe = plt.subplots(figsize=(7.4, 3.6))
axe.plot(p, -np.log(p), label='patient absent (y = 1)', color='#b5651d')
axe.plot(p, -np.log(1 - p), label='patient venu (y = 0)', color='#2b6cb0')
axe.set_xlabel('probabilité annoncée pour un absent'); axe.set_ylabel('−log(p)')
axe.set_ylim(0, 5); axe.set_title('la log-loss'); axe.legend(fontsize=8)
plt.tight_layout()
plt.show()
print('→ une annonce confiante et fausse coûte sans plafond : c’est pour cela que la')
print('  log-loss est la perte des classifieurs.')


In [ ]:
# quatre patients, calculés à la main, puis vérifiés contre la bibliothèque
from sklearn.metrics import log_loss

verites = np.array([1, 1, 0, 0])                # deux absents, deux venus
annonces = np.array([0.90, 0.05, 0.80, 0.30])   # la probabilité d’absence annoncée

logloss_main = -np.mean(np.where(verites == 1, np.log(annonces), np.log(1 - annonces)))
assert np.isclose(logloss_main, log_loss(verites, annonces)), 'la log-loss ne concorde pas'

for v, a in zip(verites, annonces):
    etat = 'absent' if v == 1 else 'venu'
    cout_log = -np.log(a) if v == 1 else -np.log(1 - a)
    print(f'  patient {etat}, annoncé {a:.2f} : −log {cout_log:5.2f}')
print(f'→ moyenne : log-loss {logloss_main:.4f} — calculée à la main, vérifiée.')


In [ ]:
# le modèle contre la probabilité constante, sur nos données
constante = np.full(len(yval), yval.mean())
print('la log-loss, sur nos données \n')
print(f'  du modèle                   : {log_loss(yval, probabilites):.4f}')
print(f'  de la probabilité constante : {log_loss(yval, constante):.4f}')
print('→ la log-loss baisse quand les probabilités annoncées sont justes,')
print('  pas quand le classement est bon. Une probabilité juste n’est pas une')
print('  probabilité calibrée : la courbe de fiabilité se trace en séance 11.')


> **Question →** `Devoir_maison_2.ipynb`, **question 10 (carnet J4)** — vos six métriques, les deux aires, et votre log-loss.


### Le coût, et le seuil

**Couche — le coût.** Ce que la décision paie, et le réglage qui la déplace.

Toutes les erreurs ne se valent pas. Une **fausse alerte** coûte **25 u.** : un créneau bloqué
pour un patient qui venait. Une **absence manquée** coûte **100 u.** : un créneau vide, une
équipe qui attend. Ces deux montants sont **fixés par l’énoncé** ; ils ne sont pas à choisir.

**Le seuil se déduit, il ne se devine pas.** Alerter quand la probabilité d’absence dépasse `p`
coûte en espérance `25 × (1 − p)`, et ne pas alerter coûte `100 × p`
absent. On alerte donc tant que `25 × (1 − p) < 100 × p`, c’est-à-dire tant que `p > 0,20`.

**Et ce seuil s’apprend sans voir la mesure.** On le choisit sur des probabilités produites par
validation croisée sur l’apprentissage : chaque ligne est prédite par un modèle qui ne l’a jamais
vue, et le jeu de contrôle ne sert qu’à vérifier.


In [ ]:
# le coût d’une décision, et la dérivation du seuil
COUT_FP, COUT_FN = 25, 100   # les deux montants de l’énoncé, en unités de coût

def cout(decision, verite=yval):
    faux_positifs = int(((decision == 1) & (verite == 0)).sum())
    faux_negatifs = int(((decision == 0) & (verite == 1)).sum())
    return COUT_FP * faux_positifs + COUT_FN * faux_negatifs

seuil_theorique = COUT_FP / (COUT_FP + COUT_FN)
print('la dérivation du seuil \n')
print(f'  pour un patient de probabilité p, alerter coûte 25 × (1 − p) ; ne pas alerter, 100 × p')
print(f'  on alerte tant que 25 × (1 − p) < 100 × p, soit p > 25 / (25 + 100) = {seuil_theorique:.2f}')
print('  → ce chiffre suppose que les probabilités annoncées veulent dire quelque chose.')


In [ ]:
# la courbe du coût, seuil par seuil — et le seuil appris, choisi hors de la mesure
from sklearn.model_selection import cross_val_predict

# le rituel : ce qu’on attend est écrit avant la mesure, sinon `noter` refuse la ligne
prevoir("le seuil de coût minimal",
        "le seuil qui coûte le moins est plus bas que 0,50",
        "une absence manquée coûte 100, une fausse alerte 25 : on alerte donc plus volontiers")

courbe = []
for seuil in np.round(np.arange(0.04, 0.96, 0.02), 2):
    decision = (probabilites > seuil).astype(int)
    courbe.append({'seuil': float(seuil), 'alertes': int(decision.sum()),
                   'exactitude (%)': round(100 * (decision == yval).mean(), 2),
                   'coût': cout(decision)})
courbe = pd.DataFrame(courbe)

# le seuil appris : chaque ligne de l’apprentissage est prédite par un modèle qui ne l’a pas vue
proba_apprises = cross_val_predict(
    HistGradientBoostingClassifier(random_state=0), Xtr, ytr, cv=5, method='predict_proba')[:, 1]
courbe_appris = pd.DataFrame([
    {'seuil': float(s), 'coût': cout((proba_apprises > s).astype(int), verite=ytr)}
    for s in np.round(np.arange(0.04, 0.96, 0.02), 2)])
seuil_appris_cout = float(courbe_appris.loc[courbe_appris['coût'].idxmin(), 'seuil'])

display(courbe.iloc[::5].reset_index(drop=True))
print(f'→ le seuil appris sur l’apprentissage seul : {seuil_appris_cout:.2f}')
print(f'  la formule                                 : {seuil_theorique:.2f}')
print(f'  le minimum de la courbe de contrôle        : {courbe.loc[courbe["coût"].idxmin(), "seuil"]:.2f}')
print('  — celui-là est choisi et jugé sur le même jeu : il gagne par construction.')

noter("le seuil de coût minimal", "la courbe du coût, seuil par seuil",
      "un seuil plus bas que 0,50", float(seuil_appris_cout))


In [ ]:
# la courbe, tracée : ce que l’apprentissage choisit, ce que le contrôle vérifie
import matplotlib.pyplot as plt

figure, axe = plt.subplots(figsize=(9.0, 4.2))
axe.plot(courbe_appris['seuil'], courbe_appris['coût'] * 1000 / len(ytr), color='#2b6cb0', linewidth=2,
         label='apprentissage (validation croisée) — choisit')
axe.plot(courbe['seuil'], courbe['coût'] * 1000 / len(yval), color='#b5651d', linewidth=2, linestyle='--',
         label='contrôle — vérifie')
axe.axvline(seuil_appris_cout, color='#2b6cb0', linestyle=':', linewidth=1)
axe.axvline(seuil_theorique, color='#4a5568', linestyle=':', linewidth=1)
axe.set_xlabel('seuil de décision'); axe.set_ylabel('coût pour 1 000 patients')
axe.set_title('le coût, seuil par seuil : la courbe qui choisit n’est pas celle qui juge')
axe.legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# le coût empilé : la part des fausses alertes, la part des absences manquées
empile = []
for seuil in courbe['seuil']:
    decision = (probabilites > seuil).astype(int)
    faux = int(((decision == 1) & (yval == 0)).sum())
    manques = int(((decision == 0) & (yval == 1)).sum())
    empile.append({'seuil': seuil, 'fausses alertes': COUT_FP * faux, 'absences manquées': COUT_FN * manques})
empile = pd.DataFrame(empile)

figure, axe = plt.subplots(figsize=(9.0, 3.8))
axe.stackplot(empile['seuil'], empile['fausses alertes'], empile['absences manquées'],
              colors=['#b5651d', '#2b6cb0'], labels=['fausses alertes (25 u. chacune)', 'absences manquées (100 u. chacune)'])
axe.axvline(seuil_appris_cout, color='#2d3748', linestyle=':', linewidth=1)
axe.set_xlabel('seuil de décision'); axe.set_ylabel('coût total')
axe.set_title('le coût total et ses deux parts, seuil par seuil')
axe.legend(fontsize=8, loc='upper center')
plt.tight_layout()
plt.show()


In [ ]:
# deux décisions, deux matrices : ce que le seuil déplace
def quatre_cases(decision):
    return {'VP': int(((decision == 1) & (yval == 1)).sum()),
            'FP': int(((decision == 1) & (yval == 0)).sum()),
            'FN': int(((decision == 0) & (yval == 1)).sum()),
            'VN': int(((decision == 0) & (yval == 0)).sum())}

a_cinquante = quatre_cases(predit)
au_seuil = quatre_cases((probabilites > seuil_appris_cout).astype(int))
comparaison = pd.DataFrame([
    {'décision': 'seuil 0,50', **a_cinquante,
     'coût pour 1 000 patients': round(cout(predit) * 1000 / len(yval))},
    {'décision': f'seuil appris {seuil_appris_cout:.2f}', **au_seuil,
     'coût pour 1 000 patients': round(cout((probabilites > seuil_appris_cout).astype(int)) * 1000 / len(yval))},
])
display(comparaison)
print(f'→ {au_seuil["VP"] - a_cinquante["VP"]} absences trouvées en plus, au prix de')
print(f'  {au_seuil["FP"] - a_cinquante["FP"]} fausses alertes en plus.')


In [ ]:
# le seuil n’est utile que s’il entre dans la fonction qui décide
def decider_au_seuil(X, seuil=seuil_appris_cout):
    return (modele.predict_proba(X)[:, 1] > seuil).astype(int)

alertes_livrees = decider_au_seuil(Xval)
print(f'→ .predict coupe à 0,50 : {int(predit.sum())} alertes.')
print(f'  decider_au_seuil coupe à {seuil_appris_cout:.2f} : {int(alertes_livrees.sum())} alertes.')
print('  Le gain mesuré n’existe que si le pipeline appelle cette fonction, pas .predict.')


**Le seuil n’est utile que s’il entre dans la fonction qui décide.** `predict` coupe à 0,50 ;
pour livrer le gain mesuré, la fonction que le pipeline appelle doit prendre le seuil appris.


> **Question →** `Devoir_maison_2.ipynb`, **question 11 (carnet J4)** — votre courbe, votre seuil, et le chiffre qui le justifie.


### La révélation : la métrique affichée, et le classement réel

**Couches — la métrique et le coût.** Ce que le classement affiché ne voit pas.

Le classement affiché se fait sur l’**exactitude**. La décision se paie en **coût**. Les deux
classements ne désignent donc pas la même tête — et c’est la seconde qu’une équipe déploierait.

L’exactitude traite les deux erreurs comme égales. Le coût pèse 100 contre 25 : il préfère
**bloquer un créneau pour rien** (25) plutôt que **laisser un créneau vide** (100). Ce n’est pas
un détail de calcul, c’est le choix de ce qu’on accepte de payer.


In [ ]:
# les deux classements, sur la même population
strategies = {'n’alerte jamais': np.zeros(len(yval), dtype=int),
              'alerte toujours': np.ones(len(yval), dtype=int)}
for seuil in (0.10, 0.20, 0.30, 0.40, 0.50):
    strategies[f'le modèle, seuil {seuil:.2f}'] = (probabilites > seuil).astype(int)
strategies[f'seuil appris ({seuil_appris_cout:.2f})'] = (probabilites > seuil_appris_cout).astype(int)

lignes = pd.DataFrame([{'stratégie': nom,
                        'exactitude (%)': round(100 * (decision == yval).mean(), 2),
                        'coût': cout(decision)}
                       for nom, decision in strategies.items()])

par_exactitude = lignes.sort_values('exactitude (%)', ascending=False).reset_index(drop=True)
par_cout = lignes.sort_values('coût').reset_index(drop=True)
print('le classement affiché (par exactitude)')
display(par_exactitude)
print('le classement réel (par coût)')
display(par_cout)


In [ ]:
# les deux classements, en pente : vert quand la stratégie gagne des places
import matplotlib.pyplot as plt

rang_exactitude = {nom: i + 1 for i, nom in enumerate(par_exactitude['stratégie'])}
rang_cout = {nom: i + 1 for i, nom in enumerate(par_cout['stratégie'])}

figure, axe = plt.subplots(figsize=(8.4, 4.8))
for nom in lignes['stratégie']:
    a, b = rang_exactitude[nom], rang_cout[nom]
    couleur = '#2f855a' if b < a else ('#c53030' if b > a else '#a0aec0')
    axe.plot([0, 1], [a, b], marker='o', color=couleur, linewidth=1.6)
    axe.text(-0.03, a, nom, ha='right', va='center', fontsize=8)
axe.set_xlim(-0.02, 1.02); axe.invert_yaxis()
axe.set_xticks([0, 1], ['rang par exactitude', 'rang par coût'])
axe.set_yticks([]); axe.set_title('le même modèle, deux classements')
plt.tight_layout()
plt.show()


In [ ]:
# ce que coûte l’erreur de métrique : le regret, ramené à 1 000 patients
vainqueur_affiche = par_exactitude.iloc[0]['stratégie']
vainqueur_reel = par_cout.iloc[0]['stratégie']
cout_affiche = int(lignes.loc[lignes['stratégie'] == vainqueur_affiche, 'coût'].iloc[0])
cout_reel = int(lignes.loc[lignes['stratégie'] == vainqueur_reel, 'coût'].iloc[0])
regret = cout_affiche - cout_reel

print(f'l’exactitude place en tête : {vainqueur_affiche}')
print(f'le coût place en tête       : {vainqueur_reel}')
print(f'→ déployer le premier coûte {regret:,} de plus que le second sur ce jeu,'.replace(',', ' '))
print(f'  soit {round(regret * 1000 / len(yval)):,} unités pour 1 000 patients.'.replace(',', ' '))


> **Question →** `Devoir_maison_2.ipynb`, **question 12 (carnet J4)** — l’écart entre la métrique affichée et le coût réel, sur **vos** prédictions.


### Quand ne pas faire de ML

**Couche — la décision.** Avant le modèle, la question du coût.

Toutes les décisions ne méritent pas un modèle. Une **règle simple écrite à la main** arrive en
une semaine. Elle ne demande aucune maintenance et aucune donnée à surveiller. Elle bat souvent
un modèle qu’il faudra réentraîner, expliquer et défendre.

**Cinq questions avant d’entraîner quoi que ce soit :**

1. une règle simple existe-t-elle déjà, et que coûte-t-elle ?
2. le modèle apporte-t-il un gain de coût mesuré, ou seulement un meilleur score ?
3. les données nécessaires existent-elles au moment de la décision ?
4. qui maintiendra le modèle quand la population changera ?
5. la décision doit-elle pouvoir s’expliquer à un patient, à un médecin, à un juge ?

Le tableau qui suit répond aux deux premières. Chaque solution est mesurée par son coût. Le
gain se lit par rapport à la référence.


In [ ]:
# l’échelle des solutions, mesurée : le coût décide, pas la mode
reference = cout(np.zeros(len(yval), dtype=int))   # ne rien faire
echelle = pd.DataFrame([
    {'solution': 'ne rien faire (la référence)',
     'exactitude (%)': round(100 * (yval == 0).mean(), 2), 'coût': reference},
    {'solution': f'règle du délai : absent si le délai dépasse {seuil_appris} jours',
     'exactitude (%)': round(100 * (regle == yval).mean(), 2), 'coût': cout(regle)},
    {'solution': 'règle de l’historique : absent si ce patient a déjà manqué un rendez-vous',
     'exactitude (%)': round(100 * ((Xval['taux_absence_passe'] > 0).astype(int) == yval).mean(), 2),
     'coût': cout((Xval['taux_absence_passe'] > 0).astype(int))},
    {'solution': 'la famille linéaire',
     'exactitude (%)': round(100 * lineaire.score(Xval_n, yval), 2),
     'coût': cout(lineaire.predict(Xval_n))},
    {'solution': 'le modèle (arbres entraînés par corrections)',
     'exactitude (%)': round(100 * exactitude, 2), 'coût': cout(predit)},
])
echelle['gain vs référence (%)'] = (100 * (reference - echelle['coût']) / reference).round(1)
display(echelle)
print('→ la meilleure exactitude de cette échelle n’est pas le coût le plus bas :')
print('  le choix se fait sur la décision, et la décision se paie.')
print('  Trois situations, dont une du sujet du cours : c’est ce que le devoir vous demande.')


**Une démonstration, sur un autre jeu.** Des capteurs de machine enregistrent la température,
la vitesse, le couple et l’usure ; 3,4 % des lignes signalent une panne. Une règle de deux lignes
— le produit du couple et de l’usure — est comparée à un modèle qui lit les cinq capteurs.
Les deux sont jugés sur le même jeu de contrôle, avec les mêmes prix.

Le résultat n’est pas celui qu’on attend : la règle est précise, mais elle laisse passer deux
pannes sur trois. C’est la mesure qui tranche, et c’est pour cela qu’on la fait.


In [ ]:
# une règle de deux lignes contre un modèle à cinq capteurs
from sklearn.metrics import f1_score, recall_score
from sklearn.model_selection import train_test_split

# le fichier accompagne le jeu de données ; à défaut, il est à côté du dossier des données
chemin_pannes = os.path.join(DOSSIER, 'machine_failure.csv')
if not os.path.exists(chemin_pannes):
    chemin_pannes = os.path.join(os.path.dirname(DOSSIER), 'machine_failure.csv')
# si le fichier est ajouté comme entrée séparée (Add Input), il vit sous /kaggle/input
if not os.path.exists(chemin_pannes) and os.path.isdir('/kaggle/input'):
    for _racine, _, _fichiers in os.walk('/kaggle/input'):
        if 'machine_failure.csv' in _fichiers:
            chemin_pannes = os.path.join(_racine, 'machine_failure.csv')
            break
pannes = pd.read_csv(chemin_pannes)
capteurs = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]',
            'Torque [Nm]', 'Tool wear [min]']
Xa_p, Xd_p, ya_p, yd_p = train_test_split(pannes[capteurs], pannes['Machine failure'],
                                          test_size=0.3, random_state=0,
                                          stratify=pannes['Machine failure'])

# la règle, écrite à la main, et jugée sur le même contrôle
regle_pannes = ((Xd_p['Torque [Nm]'] * Xd_p['Tool wear [min]']) > 11000).astype(int)
# le modèle, sur les cinq capteurs
modele_pannes = HistGradientBoostingClassifier(random_state=0).fit(Xa_p, ya_p)
predit_pannes = modele_pannes.predict(Xd_p)

from sklearn.metrics import precision_score

def bilan(nom, decision):
    manques = int(((decision == 0) & (yd_p == 1)).sum())
    fausses = int(((decision == 1) & (yd_p == 0)).sum())
    print(f'  {nom:<26} F1 {100 * f1_score(yd_p, decision):5.1f} % · rappel {100 * recall_score(yd_p, decision):5.1f} %'
          f' · précision {100 * precision_score(yd_p, decision):5.1f} %'
          f' · coût {COUT_FP * fausses + COUT_FN * manques:5d}')

print(f'  lignes : {len(pannes)}, pannes : {int(pannes["Machine failure"].sum())} ({100 * pannes["Machine failure"].mean():.1f} %)')
bilan('la règle écrite à la main', regle_pannes)
bilan('le modèle, cinq capteurs', predit_pannes)
print(f'→ la règle est précise, mais elle laisse passer {100 - 100 * recall_score(yd_p, regle_pannes):.0f} %')
print('  des pannes : elle coûte plus cher que le modèle.')
print('  La mesurer avant de l’opposer était le point de la démonstration.')


> **Question →** `Devoir_maison_2.ipynb`, **question 13 (carnet J4)** — trois situations, et ce que vous feriez **avant** d’entraîner un modèle.


### Et quand la cible est un nombre ?

**Couche — la métrique.** Les mêmes questions, pour une cible continue.

**Une tâche, deux familles.** Jusqu’ici la cible était une **classe** : le patient vient, ou non.
C’est de la **classification**. Quand la cible est une **quantité** — un prix, une durée, un nombre
de rendez-vous — la tâche change de nom : c’est de la **régression**. Les deux se mesurent
différemment, et le choix de la métrique en dépend.

**Le piège du mot, à écarter tout de suite.** Vous avez déjà vu un modèle qui porte ce nom : la
**régression logistique** de la séance 2. C’est un modèle de régression au sens statistique, mais
il **classe** — le nom du modèle ne dit pas la tâche. Une régression logistique fait de la
classification ; une régression linéaire fait de la régression.

**Et le modèle, lui, arrive plus tard.** Ici, on n’entraîne aucun modèle de régression : on mesure
une prévision naïve, la moyenne du même jour de la semaine. Le modèle linéaire de régression est
en séance 7.

Une exactitude n’existe pas pour un nombre. La **table d’activité** donne une ligne par clinique
et par jour : la date, la clinique, le jour de la semaine, et le nombre de rendez-vous. On suit
**une** clinique dans le temps, et on prévoit son nombre de rendez-vous quotidiens.

Le découpage est **chronologique** : on apprend sur le passé, on mesure sur la suite. Un
découpage au hasard laisserait le futur entrer dans l’apprentissage.


In [ ]:
# la table d’activité : une ligne par clinique et par jour
activite = pd.read_csv(os.path.join(DOSSIER, 'table_activite.csv'), low_memory=False)
activite['date'] = pd.to_datetime(activite['date'])

serie = activite[activite['clinique'] == 'C01'].sort_values('date').reset_index(drop=True)
coupure = int(len(serie) * 0.8)
appris, controle = serie.iloc[:coupure], serie.iloc[coupure:]

jours_fermes = int((activite['nb_rendez_vous'] == 0).sum())
print('la table d’activité \n')
print('  lignes :', len(activite), '— une par clinique et par jour')
print(f'  jours à zéro rendez-vous : {jours_fermes} ({100 * jours_fermes / len(activite):.1f} %) — la clinique est fermée')

# la baseline : la moyenne du même jour de la semaine, apprise sur le passé
moyenne_par_jour = appris.groupby('jour_semaine')['nb_rendez_vous'].mean()
prevision = controle['jour_semaine'].map(moyenne_par_jour).to_numpy()
verite = controle['nb_rendez_vous'].to_numpy()
ecarts = verite - prevision


In [ ]:
# la série, la coupure, et la prévision face à la réalité
import matplotlib.pyplot as plt

figure, (haut, bas) = plt.subplots(2, 1, figsize=(9.8, 5.6))
haut.plot(appris['date'], appris['nb_rendez_vous'], color='#2b6cb0', linewidth=0.8, label='appris (le passé)')
haut.plot(controle['date'], controle['nb_rendez_vous'], color='#b5651d', linewidth=0.8, label='contrôle (la vérité)')
haut.plot(controle['date'], prevision, color='#2d3748', linewidth=1.1, linestyle='--', label='la prévision')
haut.set_title('une clinique (C01), jour par jour'); haut.legend(fontsize=8)

bas.bar(controle['date'], ecarts, color='#8a9ba8', width=1.0)
bas.set_title('les écarts, jour par jour — les jours fermés font les plus grands écarts')
bas.set_xlabel('date')
plt.tight_layout()
plt.show()


| Métrique | Formule | Lecture en rendez-vous | Quand l’utiliser |
|---|---|---|---|
| **MAE** | moyenne des écarts absolus | « on se trompe de X rendez-vous en moyenne » | la lecture la plus simple, pour une équipe |
| **RMSE** | racine de la moyenne des erreurs au carré | comme la MAE, mais une grosse erreur pèse plus | quand une erreur énorme coûte plus que plusieurs petites |
| **RMSLE** | la même chose, sur les logarithmes | l’erreur relative, sans unité | quand les ordres de grandeur varient (petites et grandes cliniques) |
| **R²** | 1 − variance des erreurs / variance de la vérité | « la part de la variation expliquée », négatif si le modèle fait pire que la moyenne | pour comparer deux modèles sur la même série |
| **MAPE** | moyenne des écarts rapportés à la vérité, en % | « on se trompe de X % » | **jamais** quand la vérité peut valoir zéro |


### La démonstration, métrique par métrique (cible numérique)

Sur notre contrôle : la clinique **C01**, 219 jours. La prévision est la moyenne du même jour
de la semaine.

**MAE — erreur absolue moyenne.**

$$MAE = \frac{1}{N}\sum |\text{valeur vraie} - \text{valeur prévue}|$$

Ce qu'elle calcule : l'écart moyen, dans l'unité de la cible. Ici, des rendez-vous.
Quand c'est la bonne métrique : les erreurs de tailles voisines coûtent à peu près pareil, et
une équipe veut une lecture simple.
Cas réel : une cantine prévoit le nombre de repas. Un plat en trop et un plat manquant coûtent
presque la même chose. La MAE dit « on se trompe de douze repas en moyenne ».
Sur notre contrôle : **13,59 rendez-vous**.

**RMSE — racine de l'erreur quadratique moyenne.**

$$RMSE = \sqrt{\frac{1}{N}\sum (\text{valeur vraie} - \text{valeur prévue})^2}$$

Ce qu'elle calcule : un écart moyen qui **punit les grosses erreurs**. Elle est toujours
supérieure ou égale à la MAE.
Quand c'est la bonne métrique : une erreur énorme coûte plus que plusieurs petites.
Cas réel : la consommation électrique d'un quartier. Un pic mal prévu provoque une coupure ;
plusieurs petits écarts se compensent. On veut que le modèle ne rate jamais le pic.
Sur notre contrôle : **16,14** contre 13,59 pour la MAE. L'écart entre les deux mesure le poids
des grosses erreurs.

**RMSLE — racine de l'erreur quadratique moyenne des logarithmes.**

$$RMSLE = \sqrt{\frac{1}{N}\sum \left(\log(1+\text{valeur vraie}) - \log(1+\text{valeur prévue})\right)^2}$$

Ce qu'elle calcule : une erreur **relative**, sans unité, qui traite pareil une petite et une
grande valeur.
Quand c'est la bonne métrique : les ordres de grandeur varient beaucoup d'une ligne à l'autre.
Cas réel : les ventes de boutiques de tailles très différentes. Cinq unités d'écart n'ont pas le
même sens pour une boutique à dix et pour une à mille.
Sur notre contrôle : **0,254**.

**MAPE — erreur absolue moyenne en pourcentage.**

$$MAPE = \frac{100}{N}\sum \frac{|\text{valeur vraie} - \text{valeur prévue}|}{\text{valeur vraie}}$$

Ce qu'elle calcule : l'erreur en pourcentage, donc lisible sans connaître l'unité.
Quand c'est la bonne métrique : la cible ne vaut **jamais zéro** et on veut parler en pourcentage.
Cas réel : le chiffre d'affaires d'un magasin. « On se trompe de 8 % » se comprend sans savoir
si le magasin fait cent mille ou dix millions.
Le piège : une valeur vraie nulle fait diviser par zéro. Ici, **31 jours** de contrôle valent
zéro rendez-vous.
Sur notre contrôle, jours ouverts : **22,9 %**.

**R² — coefficient de détermination.**

$$R^2 = 1 - \frac{\text{variance des erreurs}}{\text{variance de la vérité}}$$

Ce qu'elle calcule : la part de la variation de la cible que la prévision explique. 1 = parfaite ;
0 = aussi bonne que la moyenne ; négatif = pire que la moyenne.
Quand c'est la bonne métrique : comparer deux modèles sur la même série, pour savoir lequel suit
le mieux les variations.
Cas réel : deux modèles de fréquentation d'un musée. Le R² dit lequel explique le mieux les
variations, pas lequel se trompe le moins en visiteurs.
Sur notre contrôle : **0,645**, soit environ deux tiers de la variation expliquée.

**La règle à retenir.** Choisir une métrique de régression, c'est choisir deux choses : l'unité
dans laquelle on parle de l'erreur, et le poids donné aux grosses erreurs.


In [ ]:
# la même prévision, jugée par cinq métriques
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error, r2_score

ouverts = verite > 0
mae = mean_absolute_error(verite, prevision)
rmse = mean_squared_error(verite, prevision) ** 0.5
rmsle = mean_squared_log_error(verite, prevision) ** 0.5
r2 = r2_score(verite, prevision)
mape = 100 * (abs(ecarts[ouverts]) / verite[ouverts]).mean()
print(f'  MAE   erreur moyenne, en rendez-vous  : {mae:6.2f}')
print(f'  RMSE  punit les grosses erreurs       : {rmse:6.2f}')
print(f'  RMSLE erreur relative, en logarithme  : {rmsle:6.3f}')
print(f'  R²    part de la variance expliquée   : {r2:6.3f}')
print(f'  MAPE  erreur relative, jours ouverts  : {mape:6.1f} %')
print()
print(f'→ {int((~ouverts).sum())} jours de contrôle valent zéro rendez-vous : le MAPE y')
print('  diviserait par zéro. Il faut donc une erreur qui se compare au naïf, pas')
print('  au zéro — c’est le MASE, en séance 6.')


In [ ]:
# ce que le MAPE ferait sur les jours fermés — la division par zéro, montrée
fermes = verite == 0
print(f'  jours fermés dans le contrôle : {int(fermes.sum())} sur {len(verite)}')
print(f'  un MAPE calculé sur tous les jours diviserait par zéro {int(fermes.sum())} fois.')
print('  → c’est pour cela que la ligne du MAPE précise « jours ouverts ».')

figure, axe = plt.subplots(figsize=(5.6, 2.4))
axe.bar(['jours ouverts', 'jours fermés'], [int((~fermes).sum()), int(fermes.sum())], color=['#2b6cb0', '#c53030'])
axe.set_title('les jours du contrôle : le MAPE ne peut pas lire les fermés')
axe.set_ylabel('jours')
plt.tight_layout()
plt.show()


### Ce qu’on retient

**Les trois couches, en une page.**

| Couche | Ce qu’elle règle | Ce que le carnet en a fait |
|---|---|---|
| **la perte** | ce que le modèle optimise pendant l’apprentissage | la log-loss, tracée, vérifiée sur quatre patients et comparée à la constante |
| **la métrique** | ce que le rapport affiche | six métriques depuis une seule matrice, et les deux aires qui jugent le rang |
| **le coût** | ce que la décision paie | 25 et 100 unités, un seuil déduit puis appris hors de la mesure |
**Et quand la cible est un nombre** : l’exactitude n’existe pas. MAE, RMSE, RMSLE, R² et MAPE
mesurent chacune autre chose, et le MAPE divise par zéro les jours de fermeture.


**La métrique** ne dit pas si un modèle est bon : elle dit à quelle question il répond. Sur une
cible rare, l’exactitude récompense celui qui ne fait rien ; la précision et le rappel disent ce
qu’il apporte vraiment.

**Le coût** transforme la mesure en décision. Deux modèles à un demi-point d’exactitude peuvent
coûter très différemment, et le classement qui compte est celui de la facture.

**La perte** explique pourquoi le modèle se trompe comme il se trompe : elle est le seul chiffre
que l’entraînement voit, et il n’apparaît jamais dans un rapport.

**Et parfois, il ne faut pas de modèle** — mais c’est la mesure qui le dit. Une règle de deux
lignes se compare au modèle sur le même jeu, avec les mêmes prix. Sur les capteurs de machine,
elle perd. La question reste posée pour chaque décision : « quelle est la chose la plus simple
qui atteint l’objectif ? ».
